# EDA 03 — Primeira passagem pelo dataset SRAG

In [ ]:
from pathlib import Path
import platform
import sys

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from IPython.display import display


def encontrar_raiz_projeto() -> Path:
    pasta_atual = Path.cwd().resolve()

    for pasta in [pasta_atual, *pasta_atual.parents]:
        possui_src = (pasta / "src").is_dir()
        possui_configs = (pasta / "configs").is_dir()

        if possui_src and possui_configs:
            return pasta

    raise FileNotFoundError(
        "Não foi possível localizar a raiz do projeto."
    )


ROOT_DIR = encontrar_raiz_projeto()

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))


from src import config as C
from src import config as env
from src import dados as storage


env.carregar_dotenv()

print(f"Raiz do projeto: {ROOT_DIR}")
print(f"Python: {sys.version.split()[0]}")
print(f"Sistema: {platform.system()} {platform.release()}")
print(f"pandas: {pd.__version__}")

In [ ]:
print(C.resumo())

print(f"Release configurada: {C.DATA_VERSION}")
print(f"Nome do arquivo: {C.PARQUET_NAME}")
print(f"Diretório de tabelas: {C.TABLES_DIR}")
print(f"Diretório de figuras: {C.FIGURES_DIR}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


COR_PADRAO = "tab:blue"
COR_VALIDA = "tab:green"
COR_ATENCAO = "tab:orange"
COR_PROBLEMA = "tab:red"
COR_AUSENTE = "tab:gray"

CORES_GRUPOS_EDA = {
    "Estrutura": "tab:blue",
    "Candidatas principais": "tab:green",
    "Textos complementares": "tab:gray",
    "Geografia": "tab:purple",
    "Vacinação condicional": "tab:orange",
    "Contexto social": "tab:brown",
    "Tratamento e auditoria": "tab:cyan",
    "Desfecho e auditoria": "tab:red",
}

plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 10
plt.rcParams["legend.fontsize"] = 9
plt.rcParams["xtick.labelsize"] = 9
plt.rcParams["ytick.labelsize"] = 9

print("Configuração visual do Notebook 03 carregada.")

In [ ]:
COLUNAS_ESTRUTURA = [
    "nu_notific",
    "dt_notific",
    "dt_sin_pri",
    "sem_pri",
    "hospital",
    "dt_interna",
]

COLUNAS_CANDIDATAS_PRINCIPAIS = [
    "cs_sexo",
    "nu_idade_n",
    "tp_idade",
    "cs_gestant",
    "cs_raca",
    "cs_etinia",
    "cs_escol_n",
    "cs_zona",
    "ave_suino",
    "out_anim",
    "febre",
    "tosse",
    "garganta",
    "dispneia",
    "desc_resp",
    "saturacao",
    "diarreia",
    "vomito",
    "dor_abd",
    "fadiga",
    "perd_olft",
    "perd_pala",
    "puerpera",
    "cardiopati",
    "hematologi",
    "sind_down",
    "hepatica",
    "asma",
    "diabetes",
    "neurologic",
    "pneumopati",
    "imunodepre",
    "renal",
    "obesidade",
    "tabag",
]

COLUNAS_COMPLEMENTARES_TEXTO = [
    "outro_sin",
    "outro_des",
    "obes_imc",
    "out_morbi",
    "morb_desc",
    "out_antiv",
    "out_trat",
]

COLUNAS_GEOGRAFIA = [
    "sg_uf_not",
    "co_regiona",
    "co_mun_not",
    "co_pais",
    "sg_uf",
    "co_rg_resi",
    "co_mun_res",
    "sg_uf_inte",
    "co_rg_inte",
    "co_mu_inte",
]

COLUNAS_CONDICIONAIS_VACINACAO = [
    "vacina",
    "dt_ut_dose",
    "mae_vac",
    "dt_vac_mae",
    "m_amamenta",
    "dt_doseuni",
    "dt_1_dose",
    "dt_2_dose",
    "vacina_cov",
    "dose_1_cov",
    "dose_2_cov",
    "dose_ref",
    "dose_2ref",
    "dose_adic",
    "dos_re_bi",
]

COLUNAS_CONTEXTO_SOCIAL_CONDICIONAL = [
    "pac_cocbo",
    "pac_dscbo",
    "pov_ct",
    "tp_pov_ct",
    "estrang",
]

COLUNAS_TRATAMENTO_AUDITORIA = [
    "antiviral",
    "tp_antivir",
    "dt_antivir",
    "trat_cov",
    "tipo_trat",
    "dt_trt_cov",
]

COLUNAS_DESFECHO_AUDITORIA = [
    "evolucao",
    "dt_evoluca",
    "dt_encerra",
    "dt_digita",
]

COLUNAS_EDA = list(
    dict.fromkeys(
        COLUNAS_ESTRUTURA
        + COLUNAS_CANDIDATAS_PRINCIPAIS
        + COLUNAS_COMPLEMENTARES_TEXTO
        + COLUNAS_GEOGRAFIA
        + COLUNAS_CONDICIONAIS_VACINACAO
        + COLUNAS_CONTEXTO_SOCIAL_CONDICIONAL
        + COLUNAS_TRATAMENTO_AUDITORIA
        + COLUNAS_DESFECHO_AUDITORIA
    )
)

ANO_INICIAL_COORTE = 2019
ANO_FINAL_COORTE = 2026

ANOS_COORTE = list(
    range(
        ANO_INICIAL_COORTE,
        ANO_FINAL_COORTE + 1,
    )
)

print(
    "Período definido para a coorte:",
    f"{ANO_INICIAL_COORTE}-{ANO_FINAL_COORTE}",
)

print(f"Total de colunas da EDA: {len(COLUNAS_EDA)}")
print(f"Colunas de estrutura: {len(COLUNAS_ESTRUTURA)}")
print(f"Colunas candidatas: {len(COLUNAS_CANDIDATAS_PRINCIPAIS)}")
print(f"Colunas complementares de texto: {len(COLUNAS_COMPLEMENTARES_TEXTO)}")
print(f"Colunas geográficas: {len(COLUNAS_GEOGRAFIA)}")
print(f"Colunas Condicionais de Vacinação: {len(COLUNAS_CONDICIONAIS_VACINACAO)}")
print(f"Colunas Contexto Social: {len(COLUNAS_CONTEXTO_SOCIAL_CONDICIONAL)}")
print(f"Colunas de auditoria: {len(COLUNAS_TRATAMENTO_AUDITORIA)}")
print(f"Colunas de Desfecho: {len(COLUNAS_DESFECHO_AUDITORIA)}")
print(f"Total sem duplicidades: {len(COLUNAS_EDA)}")

In [ ]:
quantidade_colunas_por_grupo = pd.DataFrame(
    {
        "grupo": [
            "Estrutura",
            "Candidatas principais",
            "Textos complementares",
            "Geografia",
            "Vacinação condicional",
            "Contexto social",
            "Tratamento e auditoria",
            "Desfecho e auditoria",
        ],
        "quantidade_colunas": [
            len(COLUNAS_ESTRUTURA),
            len(COLUNAS_CANDIDATAS_PRINCIPAIS),
            len(COLUNAS_COMPLEMENTARES_TEXTO),
            len(COLUNAS_GEOGRAFIA),
            len(COLUNAS_CONDICIONAIS_VACINACAO),
            len(COLUNAS_CONTEXTO_SOCIAL_CONDICIONAL),
            len(COLUNAS_TRATAMENTO_AUDITORIA),
            len(COLUNAS_DESFECHO_AUDITORIA),
        ],
    }
)

quantidade_colunas_por_grupo = (
    quantidade_colunas_por_grupo
    .sort_values("quantidade_colunas", ascending=True)
    .reset_index(drop=True)
)

total_colunas_grupos = int(
    quantidade_colunas_por_grupo["quantidade_colunas"].sum()
)

if total_colunas_grupos != len(COLUNAS_EDA):
    raise RuntimeError(
        "A soma das colunas por grupo não corresponde "
        "ao total de colunas da EDA."
    )

cores_colunas_grupo = [
    CORES_GRUPOS_EDA[grupo]
    for grupo in quantidade_colunas_por_grupo["grupo"]
]

figura_colunas_grupo, eixo_colunas_grupo = plt.subplots(figsize=(11, 6))

barras_colunas_grupo = eixo_colunas_grupo.barh(
    quantidade_colunas_por_grupo["grupo"],
    quantidade_colunas_por_grupo["quantidade_colunas"],
    color=cores_colunas_grupo,
)

eixo_colunas_grupo.set_title("Colunas selecionadas para a EDA por grupo")
eixo_colunas_grupo.set_xlabel("Quantidade de colunas")
eixo_colunas_grupo.set_ylabel("Grupo")
eixo_colunas_grupo.set_xlim(left=0)
eixo_colunas_grupo.grid(axis="x", alpha=0.20)

for barra, quantidade in zip(
    barras_colunas_grupo,
    quantidade_colunas_por_grupo["quantidade_colunas"],
):
    eixo_colunas_grupo.text(
        barra.get_width(),
        barra.get_y() + barra.get_height() / 2,
        f" {int(quantidade)}",
        ha="left",
        va="center",
        fontsize=9,
    )

figura_colunas_grupo.text(
    0.01,
    0.01,
    f"O recorte inicial reúne {len(COLUNAS_EDA)} colunas do Parquet.",
    fontsize=8,
)

figura_colunas_grupo.tight_layout(rect=(0, 0.04, 1, 1))

plt.show()
plt.close(figura_colunas_grupo)

display(
    quantidade_colunas_por_grupo
    .sort_values("quantidade_colunas", ascending=False)
    .reset_index(drop=True)
)

In [ ]:
caminho_parquet = storage.get_parquet(force=False)

print(f"Arquivo: {caminho_parquet}")
print(f"Existe: {caminho_parquet.exists()}")
print(
    "Tamanho do arquivo: "
    f"{caminho_parquet.stat().st_size / 1024**2:,.2f} MB, "
    f"{caminho_parquet.stat().st_size:,.2f} bytes"
)

In [ ]:
arquivo_parquet = pq.ParquetFile(caminho_parquet)

schema = arquivo_parquet.schema_arrow
colunas_disponiveis = schema.names

metadados_parquet = {
    "arquivo": caminho_parquet.name,
    "versao_dados": C.DATA_VERSION,
    "numero_linhas": arquivo_parquet.metadata.num_rows,
    "numero_colunas": arquivo_parquet.metadata.num_columns,
    "numero_row_groups": arquivo_parquet.metadata.num_row_groups,
}

display(pd.DataFrame([metadados_parquet]))

In [ ]:
schema_df = pd.DataFrame(
    {
        "coluna": schema.names,
        "tipo_arrow": [str(campo.type) for campo in schema],
    }
)

display(schema_df.head(20))
print(f"Total de campos no schema: {len(schema_df)}")

In [ ]:
conjunto_colunas_disponiveis = set(colunas_disponiveis)

status_colunas = pd.DataFrame(
    {
        "coluna": COLUNAS_EDA,
        "grupo": (
            ["estrutura"] * len(COLUNAS_ESTRUTURA)
            + ["candidatas_principais"] * len(COLUNAS_CANDIDATAS_PRINCIPAIS)
            + ["complementares_texto"] * len(COLUNAS_COMPLEMENTARES_TEXTO)
            + ["geografia"] * len(COLUNAS_GEOGRAFIA)
            + ["condicionais_vacinacao"] * len(COLUNAS_CONDICIONAIS_VACINACAO)
            + ["contexto_social_condicional"] * len(COLUNAS_CONTEXTO_SOCIAL_CONDICIONAL)
            + ["tratamento_auditoria"] * len(COLUNAS_TRATAMENTO_AUDITORIA)
            + ["desfecho_auditoria"] * len(COLUNAS_DESFECHO_AUDITORIA)
        ),
    }
)

status_colunas["existe_no_parquet"] = status_colunas["coluna"].isin(conjunto_colunas_disponiveis)

display(status_colunas)

colunas_ausentes = status_colunas.loc[
    ~status_colunas["existe_no_parquet"], "coluna"
].tolist()

if colunas_ausentes:
    print("Colunas ausentes:")
    for coluna in colunas_ausentes:
        print(f"- {coluna}")
else:
    print("Todas as colunas solicitadas existem no Parquet.")

In [ ]:
df = storage.read_srag(
    columns=COLUNAS_EDA,
    compactar=True,
    colunas_texto_livre=COLUNAS_COMPLEMENTARES_TEXTO,
)

print(f"Linhas carregadas: {df.shape[0]:,}")
print(f"Colunas carregadas: {df.shape[1]:,}")

display(df.head(0))

In [ ]:
tipos_memoria = pd.DataFrame(
    {
        "coluna": df.columns,
        "tipo_pandas": [str(df[c].dtype) for c in df.columns],
        "memoria_mb": [df[c].memory_usage(deep=True) / 1024**2 for c in df.columns],
    }
).sort_values("memoria_mb", ascending=False).reset_index(drop=True)

memoria_total_mb = df.memory_usage(deep=True).sum() / 1024**2

print(f"Memória total: {memoria_total_mb:,.2f} MB")
display(tipos_memoria)

In [ ]:
print(f"Memória total utilizada: {memoria_total_mb:,.2f} MB")

In [ ]:
display(tipos_memoria)

In [ ]:
ausencias = pd.DataFrame(
    {
        "coluna": df.columns,
        "quantidade_ausente": [df[c].isna().sum() for c in df.columns],
        "quantidade_preenchida": [df[c].notna().sum() for c in df.columns],
    }
)

ausencias["percentual_ausente"] = ausencias["quantidade_ausente"] / len(df) * 100
ausencias["percentual_preenchido"] = ausencias["quantidade_preenchida"] / len(df) * 100

ausencias = ausencias.sort_values("percentual_ausente", ascending=False).reset_index(drop=True)

display(ausencias)

In [ ]:
from matplotlib.ticker import PercentFormatter

QUANTIDADE_VARIAVEIS_AUSENCIA_GRAFICO = 100

ranking_ausencia = ausencias[
    ["coluna", "quantidade_ausente", "quantidade_preenchida", "percentual_ausente", "percentual_preenchido"]
].copy()

if "status_colunas" in globals():
    ranking_ausencia = ranking_ausencia.merge(
        status_colunas[["coluna", "grupo"]],
        on="coluna", how="left", validate="one_to_one",
    )

if "grupo" not in ranking_ausencia.columns:
    ranking_ausencia["grupo"] = "grupo_nao_informado"

ranking_ausencia["percentual_ausente"] = pd.to_numeric(
    ranking_ausencia["percentual_ausente"], errors="coerce"
)

ranking_ausencia = (
    ranking_ausencia.dropna(subset=["percentual_ausente"])
    .sort_values(["percentual_ausente", "coluna"], ascending=[False, True])
    .head(QUANTIDADE_VARIAVEIS_AUSENCIA_GRAFICO)
    .sort_values("percentual_ausente", ascending=True)
    .reset_index(drop=True)
)

def definir_cor_ausencia(percentual: float) -> str:
    if percentual >= 95:
        return "tab:red"
    if percentual >= 80:
        return "tab:orange"
    if percentual >= 50:
        return "goldenrod"
    return "tab:blue"

cores_ausencia = [definir_cor_ausencia(p) for p in ranking_ausencia["percentual_ausente"]]

altura_figura = max(8, len(ranking_ausencia) / 2)

figura_ausencia, eixo_ausencia = plt.subplots(figsize=(13, altura_figura))

barras_ausencia = eixo_ausencia.barh(
    ranking_ausencia["coluna"],
    ranking_ausencia["percentual_ausente"],
    color=cores_ausencia,
)

eixo_ausencia.set_title("Variáveis com maior percentual de ausência global")
eixo_ausencia.set_xlabel("Percentual de registros ausentes")
eixo_ausencia.set_ylabel("Variável")
eixo_ausencia.set_xlim(0, 105)
eixo_ausencia.xaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
eixo_ausencia.axvline(x=50, color="tab:gray", linestyle=":", linewidth=1)
eixo_ausencia.axvline(x=80, color="tab:orange", linestyle=":", linewidth=1)
eixo_ausencia.axvline(x=95, color="tab:red", linestyle=":", linewidth=1)
eixo_ausencia.grid(axis="x", alpha=0.15)

for barra, percentual in zip(barras_ausencia, ranking_ausencia["percentual_ausente"]):
    eixo_ausencia.text(
        barra.get_width(),
        barra.get_y() + barra.get_height() / 2,
        f" {percentual:.1f}%",
        ha="left", va="center", fontsize=8,
    )

figura_ausencia.text(
    0.01, 0.01,
    "Percentual calculado sobre a base completa. Campos condicionais devem "
    "ser reavaliados na população aplicável antes de qualquer exclusão.",
    fontsize=8,
)

figura_ausencia.tight_layout(rect=(0, 0.04, 1, 1))
plt.show()
plt.close(figura_ausencia)

display(
    ranking_ausencia.sort_values(
        ["percentual_ausente", "coluna"], ascending=[False, True]
    ).reset_index(drop=True)
)

In [ ]:
import gc

try:
    from src.features import parse_srag_date  # type: ignore[reportAttributeAccessIssue]
except ImportError:
    def parse_srag_date(serie):
        resultado = pd.to_datetime(serie, errors="coerce")
        mascara = resultado.isna() & serie.notna()
        if mascara.any():
            resultado = resultado.fillna(
                pd.to_datetime(serie, errors="coerce", format="%d/%m/%Y")
            )
        return resultado


ano_notificacao = parse_srag_date(df["dt_notific"]).dt.year
ano_internacao = parse_srag_date(df["dt_interna"]).dt.year

TAMANHO_LOTE_COLUNAS_PREENCHIMENTO = 5


def calcular_preenchimento_por_grupo(
    dados, serie_grupo, colunas, nome_grupo, tamanho_lote=5,
):
    if len(dados) != len(serie_grupo):
        raise ValueError("A base e a série de agrupamento diferem em tamanho.")
    if tamanho_lote <= 0:
        raise ValueError("O tamanho do lote precisa ser maior que zero.")

    colunas_disponiveis = [c for c in colunas if c in dados.columns]
    if not colunas_disponiveis:
        raise ValueError("Nenhuma coluna de COLUNAS_EDA está disponível.")

    serie_grupo_alinhada = pd.Series(serie_grupo, index=dados.index, name=nome_grupo)

    quantidade_por_grupo = (
        serie_grupo_alinhada
        .groupby(serie_grupo_alinhada, dropna=True, observed=True, sort=True)
        .size()
        .rename("quantidade_registros")
    )

    resultados_lotes = []

    for inicio in range(0, len(colunas_disponiveis), tamanho_lote):
        fim = min(inicio + tamanho_lote, len(colunas_disponiveis))
        colunas_lote = colunas_disponiveis[inicio:fim]

        contagem_preenchida = (
            dados[colunas_lote]
            .groupby(serie_grupo_alinhada, dropna=True, observed=True, sort=True)
            .count()
        )

        percentual_preenchido = (
            contagem_preenchida.div(quantidade_por_grupo, axis=0).mul(100)
        )

        resultados_lotes.append(percentual_preenchido)
        del contagem_preenchida, percentual_preenchido
        gc.collect()

    resultado = (
        pd.concat(resultados_lotes, axis=1)
        .reindex(columns=colunas_disponiveis)
        .rename_axis(nome_grupo)
        .reset_index()
    )

    if not resultado[colunas_disponiveis].stack().between(0, 100, inclusive="both").all():
        raise RuntimeError("Percentuais fora do intervalo 0–100%.")

    del resultados_lotes, quantidade_por_grupo, serie_grupo_alinhada
    gc.collect()

    return resultado


ano_notificacao_calculo = pd.to_numeric(ano_notificacao, errors="coerce").astype("Int16")

preenchimento_por_ano_notificacao = calcular_preenchimento_por_grupo(
    dados=df,
    serie_grupo=ano_notificacao_calculo,
    colunas=COLUNAS_EDA,
    nome_grupo="ano_notificacao",
    tamanho_lote=TAMANHO_LOTE_COLUNAS_PREENCHIMENTO,
)

display(preenchimento_por_ano_notificacao.head(20))

ano_internacao_calculo = pd.to_numeric(ano_internacao, errors="coerce").astype("Int16")

ano_internacao_estudo = ano_internacao_calculo.where(
    ano_internacao_calculo.between(
        ANO_INICIAL_COORTE, ANO_FINAL_COORTE, inclusive="both"
    )
)

preenchimento_por_ano_internacao = calcular_preenchimento_por_grupo(
    dados=df,
    serie_grupo=ano_internacao_estudo,
    colunas=COLUNAS_EDA,
    nome_grupo="ano_internacao",
    tamanho_lote=TAMANHO_LOTE_COLUNAS_PREENCHIMENTO,
)

display(preenchimento_por_ano_internacao.head(20))

del ano_notificacao_calculo, ano_internacao_calculo, ano_internacao_estudo
gc.collect()

print("Cálculo temporal concluído.")

In [ ]:
QUANTIDADE_VARIAVEIS_HEATMAP = 20
COLUNA_ANO_PREENCHIMENTO = "ano_internacao"

colunas_percentuais_preenchimento = (
    preenchimento_por_ano_internacao.columns
    .drop(COLUNA_ANO_PREENCHIMENTO)
    .tolist()
)

preenchimento_anual_matriz = (
    preenchimento_por_ano_internacao
    .set_index(COLUNA_ANO_PREENCHIMENTO)[colunas_percentuais_preenchimento]
    .apply(pd.to_numeric, errors="coerce")
    .sort_index()
)

preenchimento_anual_matriz.index = preenchimento_anual_matriz.index.astype(int)

amplitude_preenchimento = (
    preenchimento_anual_matriz.max(axis=0, skipna=True)
    .sub(preenchimento_anual_matriz.min(axis=0, skipna=True))
    .rename("amplitude_pontos_percentuais")
    .reset_index()
    .rename(columns={"index": "coluna"})
)

amplitude_preenchimento["quantidade_anos_observados"] = (
    amplitude_preenchimento["coluna"]
    .map(preenchimento_anual_matriz.notna().sum(axis=0))
    .astype("int64")
)

amplitude_preenchimento = (
    amplitude_preenchimento
    .sort_values(["amplitude_pontos_percentuais", "coluna"], ascending=[False, True])
    .reset_index(drop=True)
)

variaveis_maior_variacao = (
    amplitude_preenchimento
    .loc[amplitude_preenchimento["quantidade_anos_observados"].ge(2)]
    .head(QUANTIDADE_VARIAVEIS_HEATMAP)
    .copy()
)

colunas_heatmap_preenchimento = variaveis_maior_variacao["coluna"].tolist()

matriz_heatmap_preenchimento = (
    preenchimento_anual_matriz[colunas_heatmap_preenchimento]
    .transpose()
    .reindex(index=colunas_heatmap_preenchimento)
    .astype("float64")
)

figura_heatmap_preenchimento, eixo_heatmap_preenchimento = plt.subplots(figsize=(13, 10))

sns.heatmap(
    matriz_heatmap_preenchimento,
    annot=True, fmt=".1f", cmap="Blues", vmin=0, vmax=100,
    linewidths=0.5, linecolor="white",
    cbar_kws={"label": "Preenchimento (%)"},
    annot_kws={"fontsize": 7},
    ax=eixo_heatmap_preenchimento,
)

eixo_heatmap_preenchimento.set_title("Variáveis com maior mudança anual de preenchimento")
eixo_heatmap_preenchimento.set_xlabel("Ano da internação")
eixo_heatmap_preenchimento.set_ylabel("Variável")
eixo_heatmap_preenchimento.set_xticklabels(
    eixo_heatmap_preenchimento.get_xticklabels(), rotation=0,
)
eixo_heatmap_preenchimento.set_yticklabels(
    eixo_heatmap_preenchimento.get_yticklabels(), rotation=0,
)

figura_heatmap_preenchimento.text(
    0.01, 0.01,
    "Seleção automática das 20 maiores amplitudes anuais. Percentuais "
    "calculados sobre os registros de cada ano. O ano de 2026 possui "
    "cobertura parcial.",
    fontsize=8,
)

figura_heatmap_preenchimento.tight_layout(rect=(0.0, 0.04, 1.0, 1.0))
plt.show()
plt.close(figura_heatmap_preenchimento)

display(
    variaveis_maior_variacao[
        ["coluna", "amplitude_pontos_percentuais", "quantidade_anos_observados"]
    ].reset_index(drop=True)
)

In [ ]:
cardinalidade_variaveis = pd.DataFrame(
    {
        "coluna": df.columns,
        "valores_distintos_sem_na": [df[c].nunique(dropna=True) for c in df.columns],
        "valores_distintos_com_na": [df[c].nunique(dropna=False) for c in df.columns],
    }
).sort_values("valores_distintos_com_na", ascending=False).reset_index(drop=True)

display(cardinalidade_variaveis)

In [ ]:
from matplotlib.patches import Patch

QUANTIDADE_VARIAVEIS_CARDINALIDADE_GRAFICO = 40

ranking_cardinalidade = cardinalidade_variaveis[
    ["coluna", "valores_distintos_sem_na", "valores_distintos_com_na"]
].copy()

ranking_cardinalidade["valores_distintos_sem_na"] = pd.to_numeric(
    ranking_cardinalidade["valores_distintos_sem_na"], errors="coerce"
)
ranking_cardinalidade["valores_distintos_com_na"] = pd.to_numeric(
    ranking_cardinalidade["valores_distintos_com_na"], errors="coerce"
)

ranking_cardinalidade = (
    ranking_cardinalidade
    .dropna(subset=["valores_distintos_sem_na"])
    .loc[ranking_cardinalidade["valores_distintos_sem_na"].gt(0)]
    .sort_values(["valores_distintos_sem_na", "coluna"], ascending=[False, True])
    .head(QUANTIDADE_VARIAVEIS_CARDINALIDADE_GRAFICO)
    .sort_values("valores_distintos_sem_na", ascending=True)
    .reset_index(drop=True)
)

IDENTIFICADORES_CONHECIDOS = {"nu_notific"}
COLUNAS_TEXTO_LIVRE = set(COLUNAS_COMPLEMENTARES_TEXTO)
COLUNAS_GEOGRAFICAS = set(COLUNAS_GEOGRAFIA)

def definir_cor_cardinalidade(coluna: str) -> str:
    if coluna in IDENTIFICADORES_CONHECIDOS:
        return "tab:red"
    if coluna in COLUNAS_TEXTO_LIVRE:
        return "tab:orange"
    if coluna in COLUNAS_GEOGRAFICAS:
        return "tab:purple"
    return "tab:blue"

cores_cardinalidade = [
    definir_cor_cardinalidade(c) for c in ranking_cardinalidade["coluna"]
]

figura_cardinalidade, eixo_cardinalidade = plt.subplots(figsize=(13, 9))

barras_cardinalidade = eixo_cardinalidade.barh(
    ranking_cardinalidade["coluna"],
    ranking_cardinalidade["valores_distintos_sem_na"],
    color=cores_cardinalidade,
)

eixo_cardinalidade.set_title("Variáveis com maior quantidade de valores distintos")
eixo_cardinalidade.set_xlabel("Valores distintos sem ausência, em escala logarítmica")
eixo_cardinalidade.set_ylabel("Variável")
eixo_cardinalidade.set_xscale("log")
eixo_cardinalidade.set_xlim(left=1)
eixo_cardinalidade.grid(axis="x", which="both", alpha=0.15)

for barra, quantidade in zip(barras_cardinalidade, ranking_cardinalidade["valores_distintos_sem_na"]):
    eixo_cardinalidade.text(
        barra.get_width(),
        barra.get_y() + barra.get_height() / 2,
        f" {int(quantidade):,}".replace(",", "."),
        ha="left", va="center", fontsize=8,
    )

eixo_cardinalidade.legend(
    handles=[
        Patch(facecolor="tab:red", label="Identificador"),
        Patch(facecolor="tab:orange", label="Texto complementar"),
        Patch(facecolor="tab:purple", label="Geografia"),
        Patch(facecolor="tab:blue", label="Outras variáveis"),
    ],
    title="Tipo de variável",
    loc="lower right",
)

figura_cardinalidade.tight_layout(rect=(0.0, 0.04, 1.0, 1.0))
plt.show()
plt.close(figura_cardinalidade)

display(
    ranking_cardinalidade
    .sort_values(["valores_distintos_sem_na", "coluna"], ascending=[False, True])
    .reset_index(drop=True)
)

In [ ]:
LIMITE_CARDINALIDADE = 30

colunas_categoricas_baixas = cardinalidade_variaveis.loc[
    cardinalidade_variaveis["valores_distintos_com_na"] <= LIMITE_CARDINALIDADE,
    "coluna",
].tolist()

frequencias_categoricas = []

for coluna in colunas_categoricas_baixas:
    frequencia = (
        df[coluna]
        .value_counts(dropna=False)
        .rename_axis("valor")
        .reset_index(name="quantidade")
    )
    frequencia.insert(0, "coluna", coluna)
    frequencia["percentual"] = frequencia["quantidade"] / len(df) * 100
    frequencias_categoricas.append(frequencia)

frequencias_categoricas_df = pd.concat(frequencias_categoricas, ignore_index=True)

display(frequencias_categoricas_df.head(100))

In [ ]:
if "nu_notific" in df.columns:
    identificadores_preenchidos = df["nu_notific"].notna().sum()
    identificadores_duplicados = (
        df.loc[df["nu_notific"].notna(), "nu_notific"]
        .duplicated(keep=False)
        .sum()
    )

    resumo_identificador = pd.DataFrame(
        {
            "metrica": [
                "linhas_totais",
                "identificadores_preenchidos",
                "linhas_com_identificador_duplicado",
            ],
            "valor": [len(df), identificadores_preenchidos, identificadores_duplicados],
        }
    )

    display(resumo_identificador)

In [ ]:
REGRAS_AUDITORIA_TEMPORAL = [
    {"coluna": "evolucao", "uso_nesta_eda": "rotulo", "candidata_feature_principal": False, "risco_temporal": "desfecho", "justificativa": "Fonte utilizada para construir o alvo. Não pode ser utilizada como preditora."},
    {"coluna": "dt_evoluca", "uso_nesta_eda": "auditoria", "candidata_feature_principal": False, "risco_temporal": "posterior_admissao", "justificativa": "Data de alta ou óbito, posterior ao ponto de previsão."},
    {"coluna": "dt_encerra", "uso_nesta_eda": "auditoria", "candidata_feature_principal": False, "risco_temporal": "posterior_admissao", "justificativa": "Data de encerramento do caso, posterior ao ponto de previsão."},
    {"coluna": "dt_digita", "uso_nesta_eda": "auditoria", "candidata_feature_principal": False, "risco_temporal": "administrativo", "justificativa": "Data administrativa de inclusão do registro."},
    {"coluna": "antiviral", "uso_nesta_eda": "eda_sensibilidade", "candidata_feature_principal": False, "risco_temporal": "tratamento", "justificativa": "A disponibilidade até a admissão precisa ser confirmada."},
    {"coluna": "tp_antivir", "uso_nesta_eda": "eda_sensibilidade", "candidata_feature_principal": False, "risco_temporal": "tratamento", "justificativa": "Tipo de tratamento condicionado ao uso de antiviral."},
    {"coluna": "dt_antivir", "uso_nesta_eda": "auditoria_temporal", "candidata_feature_principal": False, "risco_temporal": "tratamento", "justificativa": "Somente tratamentos iniciados até a admissão poderão ser avaliados em sensibilidade."},
    {"coluna": "trat_cov", "uso_nesta_eda": "eda_sensibilidade", "candidata_feature_principal": False, "risco_temporal": "tratamento", "justificativa": "Pode representar decisão assistencial posterior ao ponto de previsão."},
    {"coluna": "tipo_trat", "uso_nesta_eda": "eda_sensibilidade", "candidata_feature_principal": False, "risco_temporal": "tratamento", "justificativa": "Tipo de tratamento condicionado ao registro de tratamento para COVID-19."},
    {"coluna": "dt_trt_cov", "uso_nesta_eda": "auditoria_temporal", "candidata_feature_principal": False, "risco_temporal": "tratamento", "justificativa": "Somente tratamentos iniciados até a admissão poderão ser avaliados em sensibilidade."},
]

auditoria_vazamento = pd.DataFrame(REGRAS_AUDITORIA_TEMPORAL)
auditoria_vazamento["existe_no_dataframe"] = auditoria_vazamento["coluna"].isin(df.columns)

display(auditoria_vazamento)

In [ ]:
resumo_primeira_passagem = pd.DataFrame(
    {
        "metrica": [
            "versao_dados",
            "arquivo",
            "linhas_parquet",
            "colunas_parquet",
            "linhas_carregadas",
            "colunas_carregadas",
            "memoria_mb",
            "colunas_ausentes",
        ],
        "valor": [
            C.DATA_VERSION,
            caminho_parquet.name,
            arquivo_parquet.metadata.num_rows,
            arquivo_parquet.metadata.num_columns,
            df.shape[0],
            df.shape[1],
            round(memoria_total_mb, 2),
            len(colunas_ausentes),
        ],
    }
)

display(resumo_primeira_passagem)

In [ ]:
C.TABLES_DIR.mkdir(parents=True, exist_ok=True)
print(f"Tabelas serão salvas em: {C.TABLES_DIR}")

In [ ]:
duplicidades_nu_notific = (
    df.loc[
        df["nu_notific"].duplicated(keep=False),
        [
            "nu_notific", "dt_notific", "dt_sin_pri", "hospital",
            "dt_interna", "evolucao", "dt_evoluca", "dt_encerra",
        ],
    ]
    .sort_values(["nu_notific", "dt_notific", "dt_interna"])
    .reset_index(drop=True)
)

In [ ]:
tabelas_exportacao = {
    "03_eda_tipos_memoria.csv": tipos_memoria,
    "03_eda_ausencias.csv": ausencias,
    "03_eda_preenchimento_por_ano_notificacao.csv": preenchimento_por_ano_notificacao,
    "03_eda_preenchimento_por_ano_internacao.csv": preenchimento_por_ano_internacao,
    "03_eda_cardinalidade.csv": cardinalidade_variaveis,
    "03_eda_frequencias_categoricas.csv": frequencias_categoricas_df,
    "03_eda_auditoria_vazamento.csv": auditoria_vazamento,
    "03_eda_duplicidades_nu_notific.csv": duplicidades_nu_notific,
}

tabelas_exportacao_invalidas = [
    nome for nome, tabela in tabelas_exportacao.items()
    if not isinstance(tabela, pd.DataFrame)
]

if tabelas_exportacao_invalidas:
    raise TypeError(
        "Objetos de exportação que não são DataFrames: "
        f"{tabelas_exportacao_invalidas}"
    )

for nome_arquivo, tabela in tabelas_exportacao.items():
    caminho_saida = C.TABLES_DIR / nome_arquivo
    tabela.to_csv(caminho_saida, index=False, encoding="utf-8-sig")
    print("Salvo:", caminho_saida)

print("Exportação das tabelas da EDA 03 concluída.")

In [ ]:
colunas_desfecho_fora_das_candidatas = all(
    c not in COLUNAS_CANDIDATAS_PRINCIPAIS for c in COLUNAS_DESFECHO_AUDITORIA
)

tratamentos_fora_das_candidatas_principais = all(
    c not in COLUNAS_CANDIDATAS_PRINCIPAIS for c in COLUNAS_TRATAMENTO_AUDITORIA
)

verificacoes_finais = {
    "dataframe_nao_vazio": not df.empty,
    "colunas_solicitadas_presentes": not colunas_ausentes,
    "dt_notific_disponivel": "dt_notific" in df.columns,
    "dt_interna_disponivel": "dt_interna" in df.columns,
    "desfecho_fora_das_candidatas": colunas_desfecho_fora_das_candidatas,
    "tratamentos_fora_das_candidatas_principais": tratamentos_fora_das_candidatas_principais,
    "preenchimento_por_ano_internacao_criado": not preenchimento_por_ano_internacao.empty,
    "diretorio_tabelas_existe": C.TABLES_DIR.exists(),
}

verificacoes_finais_df = pd.DataFrame(
    {
        "verificacao": list(verificacoes_finais.keys()),
        "resultado": list(verificacoes_finais.values()),
    }
)

display(verificacoes_finais_df)

if not all(verificacoes_finais.values()):
    falhas = [nome for nome, resultado in verificacoes_finais.items() if not resultado]
    raise RuntimeError(f"Verificações finais com falha: {falhas}")

print("Primeira passagem concluída com sucesso.")